LAB 6

In [16]:
con.execute("CREATE OR REPLACE TABLE bronze_customers AS SELECT * FROM read_parquet('bigdata/bronze/customers.parquet')")
con.execute("CREATE OR REPLACE TABLE bronze_transactions AS SELECT * FROM read_parquet('bigdata/bronze/transactions.parquet')")

con.execute("""
    CREATE OR REPLACE TABLE silver_transactions AS
    SELECT
        t.transaction_id, t.customer_id, t.amount, t.transaction_type,
        t.status, t.risk_score, t.is_fraud, t.ts,
        c.segment, c.credit_score,
        year(t.ts) AS year, month(t.ts) AS month, day(t.ts) AS day,
        dayofweek(t.ts) AS day_of_week,
        CASE WHEN t.amount < 100 THEN 'baixo'
             WHEN t.amount < 1000 THEN 'medio'
             ELSE 'alto' END AS amount_band
    FROM bronze_transactions t
    JOIN bronze_customers c ON t.customer_id = c.customer_id
""")
print(con.execute("SELECT COUNT(*) FROM silver_transactions").fetchone()[0])

100000


In [17]:
print(con.execute("""
    SELECT COUNT(*) AS silver, (SELECT COUNT(*) FROM bronze_transactions) AS bronze
    FROM silver_transactions
""").fetchdf())
print(con.execute("SELECT COUNT(*) AS sem_segmento FROM silver_transactions WHERE segment IS NULL").fetchdf())

   silver  bronze
0  100000  100000
   sem_segmento
0             0


In [18]:
print(con.execute("""
    SELECT segment, COUNT(*) AS transacoes,
           ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct
    FROM silver_transactions GROUP BY segment ORDER BY taxa_fraude_pct DESC
""").fetchdf())

     segment  transacoes  taxa_fraude_pct
0  High-Risk        9155             7.70
1   Standard       29689             2.21
2    Premium       61156             0.77


In [19]:
con.execute("COPY silver_transactions TO 'bigdata/silver/transactions_enriched.parquet' (FORMAT PARQUET)")
print("Silver salva em bigdata/silver/transactions_enriched.parquet")

Silver salva em bigdata/silver/transactions_enriched.parquet
